# 【バイオAI入門 #00】タンパク質を「見て」みる

緑色蛍光タンパク質（GFP）を例に、次の3つを行います。
1. UniProt からアミノ酸配列を取得する
2. PDB に登録された実験構造を表示する
3. AlphaFold の予測構造と見比べる

※ GPU は不要です。

In [ ]:
!pip install -q biopython py3Dmol requests

## Step 2: UniProt からアミノ酸配列を取得する

In [ ]:
import requests
from io import StringIO
from collections import Counter
from Bio import SeqIO

UNIPROT_ID = "P42212"  # GFP (Aequorea victoria)

fasta = requests.get(f"https://rest.uniprot.org/uniprotkb/{UNIPROT_ID}.fasta").text
record = SeqIO.read(StringIO(fasta), "fasta")

print(record.description)
print(f"長さ: {len(record.seq)} アミノ酸")
print(record.seq)

In [ ]:
for aa, n in Counter(record.seq).most_common(5):
    print(aa, n)

## Step 3: 実験で決められた立体構造（PDB: 1EMA）を表示する

In [ ]:
import py3Dmol

view = py3Dmol.view(query="pdb:1EMA", width=600, height=450)
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

## Step 4: AlphaFold の予測構造と比べる

色は信頼度 pLDDT を表します（青＝高い、赤＝低い）。

In [ ]:
meta = requests.get(f"https://alphafold.ebi.ac.uk/api/prediction/{UNIPROT_ID}").json()[0]
pdb_text = requests.get(meta["pdbUrl"]).text

view = py3Dmol.view(width=600, height=450)
view.addModel(pdb_text, "pdb")
# AlphaFold の PDB ファイルでは、B-factor 列に信頼度 pLDDT（0〜100）が入っている
view.setStyle({"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}})
view.zoomTo()
view.show()